# 🤖 Бот «Лилит» ЦЕЛИКОМ на Google Colab (бесплатно, без карты)

Этот ноутбук запускает **весь бот**: разговор (LLM), голос (Piper), картинки (ComfyUI на GPU T4) — всё в одном месте. Нужен только **Google-аккаунт** (карта НЕ нужна).

## Как пользоваться (3 минуты):
1. Откройте [colab.research.google.com](https://colab.research.google.com) → **Файл → Загрузить блокнот** → выберите этот файл.
2. Меню **Среда выполнения → Сменить среду выполнения** → **T4 GPU** → **Сохранить**.
3. Запустите все ячейки по очереди (Shift+Enter) или **Среда выполнения → Выполнить всё**.
4. Когда спросит токен — вставьте токен от @BotFather.
5. Готово! Бот работает. Идите в Telegram и общайтесь.

## ⚠️ Честно про ограничения:
- Сессия Colab живёт **несколько часов** (бесплатно), потом прерывается — запустите ноутбук заново (2 минуты).
- При бездействии ~90 минут сессия отключается сама — бот не отвечает, пока не перезапустите.
- Это не 24/7, но **бесплатно и без карты** — лучший вариант без банковской карты.
- Когда закончили — **Среда выполнения → Прервать**, чтобы не тратить лимит GPU.

In [ ]:
# ⚡ АВТОЗАПУСК: ключи запоминаются, всё остальное делает ноутбук.
# Первый раз: введите ключи — сохранятся в секреты Colab.
# Дальше: просто «Выполнить всё» — ключи подхватятся сами.
import os, sys, getpass, time
try:
    from google.colab import userdata
    TG = userdata.get('LILITH_TELEGRAM_TOKEN') or ''
    CV = userdata.get('LILITH_CIVITAI_TOKEN') or ''
except Exception:
    TG = CV = ''
if not TG:
    TG = getpass.getpass('Токен Telegram от @BotFather: ').strip()
    try:
        userdata.set('LILITH_TELEGRAM_TOKEN', TG)
    except Exception:
        pass
if not CV:
    CV = getpass.getpass('API-ключ civitai.com (Enter чтобы пропустить): ').strip()
    if CV:
        try:
            userdata.set('LILITH_CIVITAI_TOKEN', CV)
        except Exception:
            pass
print('✅ Ключи готовы:', 'Telegram ✓' if TG else 'Telegram ✗', '|', 'Civitai ✓' if CV else 'Civitai —')
# Сохраняем в файл для остальных ячеек
open('/content/.lilith_keys', 'w').write(f'{TG}\n{CV}')


In [ ]:
# 0. Проверяем GPU и память
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil
print(f"RAM: {psutil.virtual_memory().total // (1024**3)} ГБ")

In [ ]:
# 1. Токен бота — автосохранение в секреты Colab (заполняется один раз)
import getpass, os
try:
    from google.colab import userdata
    TOKEN = userdata.get('LILITH_TELEGRAM_TOKEN')
    if TOKEN:
        print('✅ Токен найден в секретах Colab')
except Exception:
    TOKEN = None
if not TOKEN:
    TOKEN = getpass.getpass('Вставьте токен от @BotFather: ').strip()
    try:
        from google.colab import userdata
        userdata.set('LILITH_TELEGRAM_TOKEN', TOKEN)
        print('✅ Токен сохранён в секреты Colab — больше вводить не нужно')
    except Exception:
        print('⚠️ Не удалось сохранить в секреты — токен будет введён заново в следующий раз')
assert TOKEN, 'Токен не введён!'
print('Токен принят ✅')


In [ ]:
# 2. Скачиваем проект и ставим зависимости (~2-3 минуты)
import os
if not os.path.exists('/content/project-lady'):
    !git clone https://github.com/samagon90/project-lady.git /content/project-lady
os.chdir('/content/project-lady')
!git checkout v0.2.9 2>/dev/null || true
!pip install -q -e . 2>&1 | tail -1
print("Проект готов ✅")

In [ ]:
# 3. Устанавливаем Ollama (мозг) и скачиваем модель
import os, subprocess, time, shutil
# Colab требует zstd для распаковки Ollama
!apt-get install -y -q zstd 2>&1 | tail -1
!curl -fsSL https://ollama.com/install.sh | sh
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH','')
# Проверяем, что ollama реально установилась
if shutil.which('ollama') is None:
    print('❌ Ollama не установилась. Пробую ещё раз с zstd...')
    !curl -fsSL https://ollama.com/install.sh | sh
if shutil.which('ollama') is None:
    print('❌ Ollama так и не установилась. Лог установки:')
    !cat /tmp/ollama_install.log 2>/dev/null || echo 'нет лога'
    raise SystemExit('Ollama не установлена — остановка')
# Запускаем Ollama через nohup — не умирает после ячейки
!nohup ollama serve > /content/ollama.log 2>&1 &
time.sleep(8)
# Ждём, пока Ollama реально поднимется (проверка через ollama list)
for i in range(15):
    r = subprocess.run(['ollama','list'], capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ Ollama работает!")
        break
    time.sleep(4)
else:
    print("❌ Ollama не поднялась. Лог:")
    print(open('/content/ollama.log').read()[-2000:])
import psutil
ram_gb = psutil.virtual_memory().total // (1024**3)
model = 'qwen2.5:3b' if ram_gb < 12 else 'dolphin-llama3:8b'
print(f"RAM {ram_gb} ГБ -> модель {model}")
!ollama pull {model}
!ollama pull nomic-embed-text
print("Ollama и модели готовы ✅")


In [ ]:
# 4. Настраиваем .env бота
import os
os.chdir('/content/project-lady')
env = f"""TELEGRAM_TOKEN={TOKEN}
LLM_MODEL={model}
LLM_BASE_URL=http://127.0.0.1:11434
EMBEDDING_MODEL=nomic-embed-text
TTS_ENABLED=false
COMFYUI_BASE_URL=http://127.0.0.1:8188
COMFYUI_CHECKPOINT=majicmixRealistic_v7.safetensors
COMFYUI_NSFW_CHECKPOINT=majicmixRealistic_v7.safetensors
DATA_DIR=/content/project-lady/data
TEMP_DIR=/content/project-lady/data/tmp
"""
open('.env','w').write(env)
!mkdir -p data/tmp
print(".env настроен ✅")

In [ ]:
# 5. Устанавливаем Piper (голос) — опционально, если нужно голосовое
!apt-get install -y -q ffmpeg 2>&1 | tail -1
!pip install -q piper-tts 2>&1 | tail -1
!mkdir -p /content/project-lady/models/piper
import os
voice = '/content/project-lady/models/piper/ru_RU-irina-medium.onnx'
if not os.path.exists(voice):
    !curl -L -o "{voice}" "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ru/ru_RU/irina/medium/ru_RU-irina-medium.onnx"
    !curl -L -o "{voice}.json" "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ru/ru_RU/irina/medium/ru_RU-irina-medium.onnx.json"
print("Голос готов (если скачался) ✅")

In [ ]:
# 5.5. Telegram Mini App (веб-интерфейс Лилит) — ОПЦИОНАЛЬНО
import os, subprocess, threading, time, re, urllib.request
answer = input('Включить Mini App (веб-интерфейс профиля/настроек)? Да/нет [да]: ').strip().lower()
if answer not in ('нет', 'no', 'n', '0'):
    print('Запускаю туннель для Mini App (localtunnel, порт 8001)...')
    os.environ.setdefault('PATH', '/usr/local/bin:' + os.environ.get('PATH', ''))
    def tunnel():
        subprocess.run(['npx', '-y', 'localtunnel', '--port', '8001'], capture_output=False)
    t = threading.Thread(target=tunnel, daemon=True)
    t.start()
    print('Ищу адрес туннеля (до 60 сек)...')
    url = None
    for _ in range(30):
        time.sleep(2)
        try:
            out = subprocess.run(['npx', '-y', 'localtunnel', '--port', '8001', '--print-requests'],
                                 capture_output=True, text=True, timeout=3)
        except Exception:
            pass
        # localtunnel печатает адрес в stderr/stdout; ищем в логах процесса
        break
    # Простой способ: спросить у пользователя адрес, если туннель не определился
    print()
    print('Туннель запущен. Узнайте его адрес так:')
    print('  В выводе выше (или в соседней ячейке) найдите строку вида: https://xxxx.loca.lt')
    print('  Если адреса нет — выполните в НОВОЙ ячейке:  !npx -y localtunnel --port 8001')
    print('  и скопируйте адрес https://xxx.loca.lt из вывода.')
    manual = input('Вставьте адрес мини-приложения (https://xxx.loca.lt) или Enter чтобы пропустить: ').strip()
    if manual.startswith('http'):
        url = manual
    if url:
        # Прописываем в .env
        env_path = '/content/project-lady/.env'
        env = open(env_path).read()
        if 'MINIAPP_HOST' not in env:
            env += '\nMINIAPP_HOST=0.0.0.0\nMINIAPP_PORT=8001\n'
        lines = [l for l in env.splitlines() if not l.startswith('WEBAPP_URL=')]
        lines.append(f'WEBAPP_URL={url}')
        open(env_path, 'w').write('\n'.join(lines) + '\n')
        print(f'✅ Mini App настроен: {url}')
        print('  Команда /app в Telegram откроет приложение.')
    else:
        print('Mini App пропущен — можно настроить позже.')
else:
    print('Mini App пропущен.')


In [ ]:
# 6. Устанавливаем ComfyUI (картинки) на GPU
import os, getpass
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt 2>&1 | tail -1
ckpt_dir = '/content/ComfyUI/models/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
ckpt = f'{ckpt_dir}/UnstableDiffusion_ema_pruned.safetensors'
# Опционально: API-ключ civitai (вставьте, если есть — скачивание надёжнее)
# Вводится один раз, в файл ноутбука НЕ сохраняется.
try:
    from google.colab import userdata
    civitai_token = userdata.get('LILITH_CIVITAI_TOKEN') or ''
    if civitai_token:
        print('✅ API-ключ civitai найден в секретах Colab')
except Exception:
    civitai_token = ''
if not civitai_token:
    civitai_token = getpass.getpass('API-ключ civitai.com (необязательно, Enter чтобы пропустить): ').strip()
    if civitai_token:
        try:
            from google.colab import userdata
            userdata.set('LILITH_CIVITAI_TOKEN', civitai_token)
            print('✅ Ключ civitai сохранён в секреты Colab')
        except Exception:
            pass
# Модель должна быть НАСТОЯЩЕЙ и БОЛЬШОЙ (~2 ГБ). Если файл пустой
# или битый — удаляем и пробуем следующий источник.
def valid_ckpt(path):
    try:
        return os.path.exists(path) and os.path.getsize(path) > 1500 * 1024 * 1024
    except OSError:
        return False
sources = []
if civitai_token:
    sources.append(('civitai.com Unstable Diffusion NSFW (с вашим API-ключом)', f'https://civitai.com/api/download/models/91623?token={civitai_token}'))
sources += [
    ('civitai.com Unstable Diffusion NSFW', 'https://civitai.com/api/download/models/91623'),
    ('HuggingFace lllyasviel', 'https://huggingface.co/lllyasviel/fav_models/resolve/main/fav/majicmixRealistic_v7.safetensors'),
    ('HuggingFace digiplay', 'https://huggingface.co/digiplay/majicMIX_realistic_v7/resolve/main/majicmixRealistic_v7.safetensors'),
    ('Stable Diffusion 1.5 (надёжный запасной)', 'https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors'),
]
ok = valid_ckpt(ckpt)
if not ok:
    if os.path.exists(ckpt):
        print('⚠️ Файл модели битый/пустой — удаляю и качаю заново')
        os.remove(ckpt)
    for name, url in sources:
        print(f'Скачиваю {name}... (~2 ГБ)')
        !curl -L --fail --max-time 3600 -o "{ckpt}" "{url}"
        if valid_ckpt(ckpt):
            print(f'✅ Модель скачана ({name}):', os.path.getsize(ckpt)//(1024**3), 'ГБ')
            ok = True
            break
        else:
            print('⚠️ Не получилось — файл маленький/битый, пробую следующий')
            if os.path.exists(ckpt):
                os.remove(ckpt)
if not ok:
    raise SystemExit('❌ Не удалось скачать модель. Проверьте интернет или скачайте вручную.')
print('ComfyUI готов ✅')

# Устанавливаем IPAdapter (твёрдый референс аватара Лилит)
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus 2>/dev/null || echo 'IPAdapter уже есть'
!pip install -q insightface onnxruntime 2>&1 | tail -1
os.makedirs('/content/ComfyUI/models/ipadapter', exist_ok=True)
# Скачиваем модель IPAdapter (plus, ~2.5 ГБ) — для SD1.5
ipa = '/content/ComfyUI/models/ipadapter/ip-adapter-plus_sd15.safetensors'
if not os.path.exists(ipa):
    !curl -L --fail --max-time 3600 -o "{ipa}" "https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus_sd15.safetensors"
print('IPAdapter готов ✅')
# Копируем аватар Лилит в input ComfyUI (референс)
os.makedirs('/content/ComfyUI/input', exist_ok=True)
!cp /content/project-lady/assets/lilith_avatar.png /content/ComfyUI/input/lilith_ref.png 2>/dev/null || echo 'аватар не найден'
print('Референс Лилит скопирован ✅')


In [ ]:
# 7. Запускаем ComfyUI на GPU (в фоне)
import subprocess, time, urllib.request
log = open('/content/comfyui.log','w')
proc = subprocess.Popen(['python','/content/ComfyUI/main.py','--listen','127.0.0.1','--port','8188'], stdout=log, stderr=log)
print("ComfyUI запускается...")
for _ in range(60):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print("✅ ComfyUI работает на GPU!")
        break
    except Exception:
        pass
else:
    print("ComfyUI не поднялся — смотрите лог:")
    print(open('/content/comfyui.log').read()[-2000:])

In [ ]:
# 8. Запускаем бота! (работает, пока открыта сессия)
import os, subprocess, time
os.chdir('/content/project-lady')
!python -m alembic upgrade head 2>&1 | tail -1
# Проверяем, что Ollama жива (если нет — перезапускаем)
r = subprocess.run(['ollama','list'], capture_output=True, text=True)
if r.returncode != 0:
    print("Ollama не запущена — запускаю заново...")
    !nohup ollama serve > /content/ollama.log 2>&1 &
    time.sleep(10)
bot_proc = subprocess.Popen(['python','-m','src.main'], stdout=open('/content/bot.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
log = open('/content/bot.log').read()
print(log[-1500:])
print("\n✅ Бот запущен! Идите в Telegram и напишите /start")


## 📋 Шпаргалка

- **Остановить бота**: Среда выполнения → Прервать.
- **Перезапустить после обрыва**: откройте ноутбук → Среда выполнения → Выполнить всё → вставить токен → готово.
- **Логи**: `print(open('/content/bot.log').read()[-2000:])` — вставьте в новую ячейку и запустите.
- **Сменить модель**: в ячейке 3 поменяйте `model = ...` на `dolphin-llama3:8b` или `qwen2.5:7b`.

## ⚠️ Важно
- Без карты и бесплатно — это лучший вариант, но сессии не вечные. Для 24/7 нужен платный VPS или карта (Oracle).
- Если Colab пишет про лимит GPU — подождите час или используйте CPU-среду (бот будет работать, картинки медленные).
- Не закрывайте вкладку с ноутбуком, пока бот нужен.